# 🎲 Modelo Neural Profundo com ResNet para Análise da Mega-Sena v4.0

## 📋 Objetivo
Modelo **profundo e denso** com **blocos ResNet** seguidos de múltiplas camadas LSTM para análise de sequências de jogos da Mega-Sena.

### 🏗️ Arquitetura desta versão:
- 🔥 **2 Blocos ResNet** (Convolutional Neural Networks com conexões residuais)
- 🧠 **4 camadas LSTM** (Encoder + Decoder duplos)
- 🏗️ **Torre Densa Profunda** com 8 camadas
- 📊 **Input Simples**: Apenas sequência de jogos `[[jogo1], [jogo2], ..., [jogo100]]`

### 💡 O que são Blocos ResNet?
**ResNet (Residual Networks)** são redes neurais que utilizam **conexões residuais** (shortcuts) que permitem que o gradiente flua diretamente através da rede, facilitando o treinamento de redes muito profundas.

**Vantagens:**
- ✅ Resolve o problema de **gradiente desvanecente**
- ✅ Permite treinar redes **muito mais profundas**
- ✅ Melhor **extração de features** temporais
- ✅ **Preserva informações** através dos shortcuts

### 📊 Fluxo da Arquitetura:
```
Input (100, 6)
    ↓
Expansão de Features: 6 → 128 → 256
    ↓
🆕 ResNet Block 1 (256 filtros)
    ├─ Conv1D → BatchNorm → ReLU
    ├─ Conv1D → BatchNorm
    └─ Add (shortcut) → ReLU
    ↓
🆕 ResNet Block 2 (256 filtros)
    ├─ Conv1D → BatchNorm → ReLU
    ├─ Conv1D → BatchNorm
    └─ Add (shortcut) → ReLU
    ↓
Dropout (0.2)
    ↓
LSTM Encoder 1 (256 units)
    ↓
LSTM Encoder 2 (256 units)
    ↓
LSTM Decoder 1 (256 units)
    ↓
LSTM Decoder 2 (128 units)
    ↓
Torre Densa (8 camadas: 512→512→256→256→128→128→64→64)
    ↓
Output (60 probabilidades)
```


In [8]:
# ========================
# IMPORTS E CONFIGURAÇÕES
# ========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import comb
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Dropout, BatchNormalization, Add, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

# Configurações de visualização
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('seaborn-darkgrid')
sns.set_palette('viridis')

# Seeds para reprodutibilidade
np.random.seed(42)
tf.random.set_seed(42)

# Parâmetros globais
TOTAL_NUMBERS = 60      # Total de números da Mega-Sena (1-60)
NUMBERS_DRAWN = 6       # Números sorteados por jogo
BATCH_SIZE = 32         # Tamanho do batch para treinamento
EPOCHS = 200            # Número máximo de épocas

print(f"TensorFlow versão: {tf.__version__}")
print(f"GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow versão: 2.16.1
GPU disponível: False


---
## 1. 📥 Carregar Dados

Carregamos os dados pré-processados que foram gerados pelo notebook `prepare_data_simple.ipynb`.

**Estrutura dos dados:**
- `X_train`, `X_val`, `X_test`: Sequências de 100 jogos (shape: `[n_samples, 100, 6]`)
- `y_train`, `y_val`, `y_test`: Próximo jogo codificado em one-hot (shape: `[n_samples, 60]`)


In [9]:
try:
    X_train = np.load('X_train.npy')
    y_train = np.load('y_train.npy')
    X_val = np.load('X_val.npy')
    y_val = np.load('y_val.npy')
    X_test = np.load('X_test.npy')
    y_test = np.load('y_test.npy')
    
    WINDOW_SIZE = X_train.shape[1]      # 100 jogos
    INPUT_FEATURES = X_train.shape[2]   # 6 números por jogo
    
    print("✅ Dados carregados com sucesso!")
    print(f"   Treino: {X_train.shape[0]:,} amostras")
    print(f"   Validação: {X_val.shape[0]:,} amostras")
    print(f"   Teste: {X_test.shape[0]:,} amostras")
    print(f"\n   WINDOW_SIZE: {WINDOW_SIZE}")
    print(f"   INPUT_FEATURES: {INPUT_FEATURES}")
    
except FileNotFoundError:
    print("❌ ERRO: Arquivos .npy não encontrados.")
    print("   Execute primeiro o notebook 'prepare_data_simple.ipynb'")
    raise

✅ Dados carregados com sucesso!
   Treino: 1,854 amostras
   Validação: 397 amostras
   Teste: 398 amostras

   WINDOW_SIZE: 300
   INPUT_FEATURES: 6


---
## 2. 🔧 Normalização dos Dados

Normalizamos os dados dividindo por 60 (valor máximo) para que fiquem no intervalo [0, 1].
Isso ajuda o modelo a convergir mais rapidamente durante o treinamento.


In [10]:
# Normalização: dividir por 60 para ter valores entre 0 e 1
X_train_norm = (X_train / 60.0).astype(np.float32)
X_val_norm = (X_val / 60.0).astype(np.float32)
X_test_norm = (X_test / 60.0).astype(np.float32)

# Converter labels para float32
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

print("✅ Dados normalizados")
print(f"   Range de X_train_norm: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]")

✅ Dados normalizados
   Range de X_train_norm: [0.017, 1.000]


---
## 2.5 📊 Criação do Vetor de Frequências

### Nova Feature: Frequência de Aparição dos Números

Para cada amostra, vamos criar um vetor adicional de tamanho **60** que contém a quantidade de vezes que cada número (1-60) apareceu na janela de jogos.

**Estrutura:**
- `freq[0]` = quantas vezes o número 1 apareceu na janela
- `freq[1]` = quantas vezes o número 2 apareceu na janela
- ...
- `freq[59]` = quantas vezes o número 60 apareceu na janela

**Normalização:** Dividimos por 60 para manter valores entre 0 e ~5 (dependendo do tamanho da janela).

### 💡 Por que isso ajuda?
- Números que aparecem com mais frequência podem ter maior probabilidade de aparecer novamente
- Ou o contrário: números "atrasados" podem ter maior chance de sair
- O modelo pode aprender esses padrões estatísticos


In [ ]:
# ============================================================================
# CRIAÇÃO DO VETOR DE FREQUÊNCIAS PARA CADA AMOSTRA
# ============================================================================

def create_frequency_vectors(X_data):
    """
    Cria vetores de frequência para cada amostra.
    
    Para cada amostra, conta quantas vezes cada número (1-60) aparece
    na janela de jogos.
    
    Args:
        X_data: array de shape (n_samples, window_size, 6)
                Cada linha contém 6 números de 1-60
    
    Returns:
        freq_vectors: array de shape (n_samples, 60)
                     Contagem normalizada de cada número
    """
    n_samples = X_data.shape[0]
    freq_vectors = np.zeros((n_samples, 60), dtype=np.float32)
    
    print(f"Criando vetores de frequência para {n_samples} amostras...")
    
    for i in range(n_samples):
        # Achatar todos os números da janela
        all_numbers = X_data[i].flatten().astype(int)
        
        # Contar frequência de cada número (1-60)
        for num in all_numbers:
            if 1 <= num <= 60:
                freq_vectors[i, num - 1] += 1
    
    # Normalizar dividindo por 60
    freq_vectors_norm = freq_vectors / 60.0
    
    print(f"   Shape: {freq_vectors_norm.shape}")
    print(f"   Range: [{freq_vectors_norm.min():.3f}, {freq_vectors_norm.max():.3f}]")
    print(f"   Média: {freq_vectors_norm.mean():.3f}")
    
    return freq_vectors_norm


# Criar vetores de frequência para treino, validação e teste
print("="*60)
print("📊 CRIANDO VETORES DE FREQUÊNCIA")
print("="*60)

print("\n🔹 Conjunto de Treino:")
X_train_freq = create_frequency_vectors(X_train)

print("\n🔹 Conjunto de Validação:")
X_val_freq = create_frequency_vectors(X_val)

print("\n🔹 Conjunto de Teste:")
X_test_freq = create_frequency_vectors(X_test)

print("\n✅ Vetores de frequência criados com sucesso!")
print(f"\n📊 Resumo das shapes:")
print(f"   X_train_norm: {X_train_norm.shape} (sequência de jogos)")
print(f"   X_train_freq: {X_train_freq.shape} (frequência dos números)")


---
## 2.6 📉 Criação do Vetor de Lacuna (Gap para Distribuição Uniforme)

### Nova Feature: Quanto falta para cada número alcançar a distribuição igualitária

Se a distribuição fosse perfeitamente uniforme, cada número (1-60) deveria aparecer o mesmo número de vezes na janela.

**Cálculo:**
- `total_numeros_sorteados = window_size × 6` (ex: 100 jogos × 6 = 600 números)
- `esperado_por_numero = total / 60` (ex: 600 / 60 = 10 aparições esperadas)
- `gap[i] = esperado - frequência_real[i]`

**Interpretação:**
- `gap > 0`: Número está "atrasado" (apareceu menos que o esperado)
- `gap < 0`: Número está "adiantado" (apareceu mais que o esperado)
- `gap = 0`: Número está na média esperada

### 💡 Por que isso pode ajudar?
- **Teoria do equilíbrio**: Números "atrasados" podem ter maior chance de sair
- **Padrão estatístico**: O modelo pode aprender se há correlação entre gap e aparições futuras


In [ ]:
# ============================================================================
# CRIAÇÃO DO VETOR DE LACUNA (GAP) PARA DISTRIBUIÇÃO UNIFORME
# ============================================================================

def create_gap_vectors(X_data):
    """
    Cria vetores de lacuna (gap) para cada amostra.
    
    O gap indica quanto cada número está abaixo/acima da distribuição uniforme esperada.
    
    Args:
        X_data: array de shape (n_samples, window_size, 6)
    
    Returns:
        gap_vectors: array de shape (n_samples, 60)
                    Valores positivos = número atrasado
                    Valores negativos = número adiantado
    """
    n_samples = X_data.shape[0]
    window_size = X_data.shape[1]
    
    # Total de números sorteados na janela
    total_numbers = window_size * 6
    
    # Frequência esperada por número se distribuição fosse uniforme
    expected_per_number = total_numbers / 60.0
    
    gap_vectors = np.zeros((n_samples, 60), dtype=np.float32)
    
    print(f"Criando vetores de lacuna (gap) para {n_samples} amostras...")
    print(f"   Total de números por janela: {total_numbers}")
    print(f"   Frequência esperada por número: {expected_per_number:.2f}")
    
    for i in range(n_samples):
        # Contar frequência real de cada número
        freq = np.zeros(60)
        all_numbers = X_data[i].flatten().astype(int)
        
        for num in all_numbers:
            if 1 <= num <= 60:
                freq[num - 1] += 1
        
        # Calcular gap: quanto falta para atingir a distribuição uniforme
        # gap positivo = número atrasado (apareceu menos que esperado)
        # gap negativo = número adiantado (apareceu mais que esperado)
        gap_vectors[i] = expected_per_number - freq
    
    # Normalizar dividindo pela frequência esperada
    # Isso coloca os valores em uma escala relativa (-1 a +1 aproximadamente)
    gap_vectors_norm = gap_vectors / expected_per_number
    
    print(f"   Shape: {gap_vectors_norm.shape}")
    print(f"   Range: [{gap_vectors_norm.min():.3f}, {gap_vectors_norm.max():.3f}]")
    print(f"   Média: {gap_vectors_norm.mean():.6f} (deve ser ~0)")
    
    return gap_vectors_norm


# Criar vetores de gap para treino, validação e teste
print("="*60)
print("📉 CRIANDO VETORES DE LACUNA (GAP)")
print("="*60)

print("\n🔹 Conjunto de Treino:")
X_train_gap = create_gap_vectors(X_train)

print("\n🔹 Conjunto de Validação:")
X_val_gap = create_gap_vectors(X_val)

print("\n🔹 Conjunto de Teste:")
X_test_gap = create_gap_vectors(X_test)

print("\n✅ Vetores de gap criados com sucesso!")
print(f"\n📊 Resumo das shapes:")
print(f"   X_train_norm: {X_train_norm.shape} (sequência de jogos)")
print(f"   X_train_freq: {X_train_freq.shape} (frequência dos números)")
print(f"   X_train_gap:  {X_train_gap.shape} (gap para distribuição uniforme)")


In [ ]:
# ============================================================================
# CARREGAR VETORES TOP 10 E BOTTOM 10
# ============================================================================

print("\n" + "="*60)
print("📊 CARREGANDO VETORES TOP 10 E BOTTOM 10")
print("="*60)

try:
    X_train_top10 = np.load('X_train_top10.npy')
    X_train_bottom10 = np.load('X_train_bottom10.npy')
    X_val_top10 = np.load('X_val_top10.npy')
    X_val_bottom10 = np.load('X_val_bottom10.npy')
    X_test_top10 = np.load('X_test_top10.npy')
    X_test_bottom10 = np.load('X_test_bottom10.npy')
    
    print("\n✅ Vetores Top/Bottom 10 carregados!")
    print(f"   X_train_top10: {X_train_top10.shape}")
    print(f"   X_train_bottom10: {X_train_bottom10.shape}")
    
except FileNotFoundError:
    print("\n⚠️ Arquivos Top/Bottom 10 não encontrados.")
    print("   Execute prepare_data_simple.ipynb novamente!")
    
    # Criar vetores temporários
    def create_top_bottom_vectors(X_data):
        n_samples = X_data.shape[0]
        top10_vectors = np.zeros((n_samples, 60), dtype=np.float32)
        bottom10_vectors = np.zeros((n_samples, 60), dtype=np.float32)
        
        for i in range(n_samples):
            freq = np.zeros(60)
            all_numbers = X_data[i].flatten().astype(int)
            for num in all_numbers:
                if 1 <= num <= 60:
                    freq[num - 1] += 1
            sorted_indices = np.argsort(freq)
            bottom10_vectors[i, sorted_indices[:10]] = 1.0
            top10_vectors[i, sorted_indices[-10:]] = 1.0
        return top10_vectors, bottom10_vectors
    
    print("   Criando vetores temporários...")
    X_train_top10, X_train_bottom10 = create_top_bottom_vectors(X_train)
    X_val_top10, X_val_bottom10 = create_top_bottom_vectors(X_val)
    X_test_top10, X_test_bottom10 = create_top_bottom_vectors(X_test)
    print("   ✅ Vetores criados!")


---
## 3. 🔥 Implementação do Bloco ResNet

### O que é um Bloco ResNet?

Um **bloco ResNet** é composto por:
1. **Caminho Principal**: Duas camadas convolucionais com BatchNormalization
2. **Shortcut Connection**: Conexão direta do input para o output
3. **Adição**: Soma do caminho principal com o shortcut
4. **Ativação**: ReLU final

### Fórmula Matemática:
```
y = F(x) + x
```
Onde:
- `x` é o input
- `F(x)` é o resultado das convoluções
- `y` é o output final

### Por que usar ResNet?
- **Gradiente Desvanecente**: O shortcut permite que o gradiente flua diretamente
- **Identidade**: Se F(x) = 0, o bloco aprende a função identidade
- **Profundidade**: Permite treinar redes muito mais profundas


In [11]:
def resnet_block(x, filters, kernel_size=3, name_prefix="resnet"):
    """
    Bloco ResNet para dados sequenciais (1D)
    
    Args:
        x: tensor de entrada (shape: [batch, timesteps, features])
        filters: número de filtros para Conv1D
        kernel_size: tamanho do kernel de convolução
        name_prefix: prefixo para nomear as camadas
    
    Returns:
        tensor de saída com conexão residual (shape: [batch, timesteps, filters])
    
    Estrutura:
        Input
          ├─> Conv1D → BatchNorm → ReLU
          ├─> Conv1D → BatchNorm
          └─> Add (com shortcut) → ReLU → Output
    """
    # Salvar o input original (shortcut)
    shortcut = x
    
    # ==========================================
    # Caminho Principal - Primeira Convolução
    # ==========================================
    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding='same',  # Mantém o tamanho da sequência
        kernel_regularizer=l2(0.001),  # Regularização L2
        name=f"{name_prefix}_conv1"
    )(x)
    x = BatchNormalization(name=f"{name_prefix}_bn1")(x)
    x = tf.keras.layers.Activation('relu', name=f"{name_prefix}_relu1")(x)
    
    # ==========================================
    # Caminho Principal - Segunda Convolução
    # ==========================================
    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding='same',
        kernel_regularizer=l2(0.001),
        name=f"{name_prefix}_conv2"
    )(x)
    x = BatchNormalization(name=f"{name_prefix}_bn2")(x)
    
    # ==========================================
    # Ajustar Shortcut (se necessário)
    # ==========================================
    # Se o número de features mudou, precisamos ajustar o shortcut
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(
            filters=filters,
            kernel_size=1,  # Convolução 1x1 para ajustar dimensões
            padding='same',
            name=f"{name_prefix}_shortcut"
        )(shortcut)
    
    # ==========================================
    # Adicionar Conexão Residual
    # ==========================================
    x = Add(name=f"{name_prefix}_add")([shortcut, x])
    x = tf.keras.layers.Activation('relu', name=f"{name_prefix}_relu2")(x)
    
    return x

print("✅ Função resnet_block() definida")

✅ Função resnet_block() definida


---
## 4. 🧠 Construção do Modelo ResNet + Attention + LSTM

Vamos construir um modelo híbrido que combina:
1. **ResNet**: Para extração de features espaciais/temporais
2. **🎯 Multi-Head Attention**: Para capturar relações importantes entre timesteps
3. **LSTM**: Para capturar dependências temporais de longo prazo
4. **Dense Layers**: Para classificação final

### 💡 Por que usar Attention antes de cada LSTM?
- **Foco seletivo**: O mecanismo de atenção permite que o modelo "preste atenção" aos jogos mais relevantes
- **Relações de longo alcance**: Captura dependências entre jogos distantes na sequência
- **Conexões residuais**: Preservam o gradiente durante o treinamento

### 🏗️ Arquitetura Detalhada:

| Estágio | Camadas | Descrição |
|---------|---------|------------|
| **0. Input** | Input | (window_size, 6) |
| **1. Expansão** | Dense → Dense → BN | 6 → 128 → 256 features |
| **2. ResNet** | ResNet Block 1 + 2 | Extração de features |
| **3. Attn+LSTM** | MultiHeadAttention → LSTM Enc 1 | 4 heads + 256 units |
| **4. Attn+LSTM** | MultiHeadAttention → LSTM Enc 2 | 4 heads + 256 units |
| **5. Attn+LSTM** | MultiHeadAttention → LSTM Dec 1 | 4 heads + 256 units |
| **6. Attn+LSTM** | MultiHeadAttention → LSTM Dec 2 | 4 heads + 128 units |
| **7. Dense** | Torre Densa (8 camadas) | 512→512→256→256→128→128→64→64 |
| **8. Output** | Dense (sigmoid) | 60 probabilidades |


In [ ]:
def build_resnet_lstm_model(window_size, input_features):
    """
    Modelo ULTRA PROFUNDO com CINCO INPUTS:
    1. Sequência de jogos: (window_size, 6)
    2. Vetor de frequências: (60,)
    3. Vetor de gap (lacuna): (60,)
    4. Vetor Top 10 (mais frequentes): (60,)
    5. Vetor Bottom 10 (menos frequentes): (60,)
    
    Arquitetura PROFUNDA:
    - 4 Blocos ResNet
    - 8 Camadas Attention + LSTM
    - 12 Camadas Densas
    """
    print("\n" + "="*80)
    print("🧠 CONSTRUINDO MODELO ULTRA PROFUNDO (5 INPUTS)")
    print("="*80)
    
    # ==========================================
    # INPUTS
    # ==========================================
    input_sequence = Input(shape=(window_size, input_features), name="Input_Sequence")
    input_frequency = Input(shape=(60,), name="Input_Frequency")
    input_gap = Input(shape=(60,), name="Input_Gap")
    input_top10 = Input(shape=(60,), name="Input_Top10")
    input_bottom10 = Input(shape=(60,), name="Input_Bottom10")
    
    print(f"   Input 1 (Sequência): ({window_size}, {input_features})")
    print(f"   Input 2 (Frequência): (60,)")
    print(f"   Input 3 (Gap): (60,)")
    print(f"   Input 4 (Top10): (60,)")
    print(f"   Input 5 (Bottom10): (60,)")
    
    # ==========================================
    # CAMINHO 1: Processamento da Sequência (PROFUNDO)
    # ==========================================
    print("\n   [CAMINHO 1] Sequência - ResNet + Attention + LSTM:")
    
    # Expansão profunda
    x = Dense(128, activation='relu', name="Expand_1")(input_sequence)
    x = Dense(256, activation='relu', name="Expand_2")(x)
    x = Dense(512, activation='relu', name="Expand_3")(x)
    x = BatchNormalization(name="BN_Expand")(x)
    print("   [1.0] Expansão: 6 → 128 → 256 → 512")
    
    # 4 Blocos ResNet
    for i in range(4):
        x = resnet_block(x, filters=512, name_prefix=f"ResNet_{i+1}")
    x = Dropout(0.2, name="Dropout_ResNet")(x)
    print("   [1.1] 4 Blocos ResNet: 512 filtros cada")
    
    # 8 camadas Attention + LSTM
    lstm_units = [512, 512, 512, 512, 256, 256, 256, 128]
    
    for i, units in enumerate(lstm_units):
        # Multi-Head Attention
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=8, key_dim=64, name=f"Attention_{i+1}"
        )(x, x)
        attn = tf.keras.layers.Add(name=f"Attn_Res_{i+1}")([x, attn])
        attn = tf.keras.layers.LayerNormalization(name=f"Attn_LN_{i+1}")(attn)
        
        # LSTM
        return_seq = (i < len(lstm_units) - 1)  # False apenas no último
        lstm = LSTM(units, return_sequences=return_seq, name=f"LSTM_{i+1}",
                    kernel_regularizer=l2(0.001))(attn)
        
        if return_seq:
            lstm = BatchNormalization(name=f"BN_LSTM_{i+1}")(lstm)
            lstm = Dropout(0.2)(lstm)
            x = lstm
        else:
            lstm = BatchNormalization(name=f"BN_LSTM_{i+1}")(lstm)
            lstm = Dropout(0.3)(lstm)
            sequence_output = lstm
    
    print(f"   [1.2] 8 Attention + LSTM: {lstm_units}")
    
    # ==========================================
    # CAMINHO 2: Frequências
    # ==========================================
    print("\n   [CAMINHO 2] Frequências:")
    freq = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(input_frequency)
    freq = BatchNormalization()(freq)
    freq = Dropout(0.2)(freq)
    freq = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(freq)
    freq = BatchNormalization()(freq)
    freq = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(freq)
    freq = BatchNormalization()(freq)
    print("   60 → 256 → 128 → 64")
    
    # ==========================================
    # CAMINHO 3: Gap
    # ==========================================
    print("\n   [CAMINHO 3] Gap:")
    gap = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(input_gap)
    gap = BatchNormalization()(gap)
    gap = Dropout(0.2)(gap)
    gap = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(gap)
    gap = BatchNormalization()(gap)
    gap = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(gap)
    gap = BatchNormalization()(gap)
    print("   60 → 256 → 128 → 64")
    
    # ==========================================
    # CAMINHO 4: Top 10
    # ==========================================
    print("\n   [CAMINHO 4] Top 10:")
    top10 = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(input_top10)
    top10 = BatchNormalization()(top10)
    top10 = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(top10)
    top10 = BatchNormalization()(top10)
    print("   60 → 128 → 64")
    
    # ==========================================
    # CAMINHO 5: Bottom 10
    # ==========================================
    print("\n   [CAMINHO 5] Bottom 10:")
    bottom10 = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(input_bottom10)
    bottom10 = BatchNormalization()(bottom10)
    bottom10 = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(bottom10)
    bottom10 = BatchNormalization()(bottom10)
    print("   60 → 128 → 64")
    
    # ==========================================
    # MERGE - Concatenação
    # ==========================================
    print("\n   [MERGE] Concatenação:")
    merged = tf.keras.layers.Concatenate(name="Merge")([
        sequence_output, freq, gap, top10, bottom10
    ])
    print("   128 + 64 + 64 + 64 + 64 = 384")
    
    # ==========================================
    # TORRE DENSA PROFUNDA (12 camadas)
    # ==========================================
    x = merged
    dense_dims = [1024, 1024, 512, 512, 512, 256, 256, 256, 128, 128, 64, 64]
    
    print(f"\n   [DENSE] Torre Profunda: {dense_dims}")
    
    for i, dim in enumerate(dense_dims):
        x = Dense(dim, activation='swish', name=f"Dense_{i+1}",
                  kernel_regularizer=l2(0.001))(x)
        x = BatchNormalization(name=f"BN_Dense_{i+1}")(x)
        if i % 2 == 0:
            x = Dropout(0.25)(x)
    
    # ==========================================
    # SAÍDA
    # ==========================================
    outputs = Dense(TOTAL_NUMBERS, activation='sigmoid', name="Output")(x)
    print("   [OUTPUT] 60 probabilidades (sigmoid)")
    
    # ==========================================
    # CRIAR MODELO
    # ==========================================
    model = Model(
        inputs=[input_sequence, input_frequency, input_gap, input_top10, input_bottom10],
        outputs=outputs,
        name="MegaSena_UltraDeep_5Inputs"
    )
    
    print("\n" + "="*80)
    print(f"   Total de camadas: {len(model.layers)}")
    print(f"   Parâmetros treináveis: {model.count_params():,}")
    print("="*80)
    
    return model


# Construir o modelo
model = build_resnet_lstm_model(WINDOW_SIZE, INPUT_FEATURES)


---
## 5. ⚙️ Compilação do Modelo

### Componentes da Compilação:

1. **Optimizer**: Adam com learning rate de 0.001
   - Adaptativo e eficiente
   - Bom para problemas complexos

2. **Loss Function**: Binary Crossentropy
   - Apropriada para classificação multi-label
   - Cada número é tratado independentemente

3. **Métricas de Monitoramento**:
   - 📈 **Accuracy**: Porcentagem de predições corretas
   - 🎯 **Precision**: Quantos dos números preditos eram corretos (TP / (TP + FP))
   - 🔍 **Recall**: Quantos dos números corretos foram encontrados (TP / (TP + FN))
   - ⚖️ **F1 Score**: Média harmônica entre Precision e Recall
   - 📊 **AUC**: Área sob a curva ROC (capacidade de discriminação)

### 💡 Por que essas métricas?
- **Precision e Recall**: Importantes para entender o equilíbrio entre falsos positivos e falsos negativos
- **F1 Score**: Combina precision e recall em uma única métrica
- **AUC**: Independente do threshold, mede a capacidade geral de classificação


In [ ]:
# ========================
# Métricas Customizadas
# ========================

def precision_metric(y_true, y_pred):
    """
    Calcula a Precision durante o treinamento
    
    Precision = TP / (TP + FP)
    
    Mede quantos dos números preditos como positivos eram realmente corretos.
    Alta precision = poucas predições falsas positivas.
    """
    y_pred_round = K.round(K.clip(y_pred, 0, 1))
    tp = K.sum(y_true * y_pred_round)
    fp = K.sum((1 - y_true) * y_pred_round)
    precision = tp / (tp + fp + K.epsilon())
    return precision


def recall_metric(y_true, y_pred):
    """
    Calcula o Recall durante o treinamento
    
    Recall = TP / (TP + FN)
    
    Mede quantos dos números que deveriam ser preditos foram encontrados.
    Alto recall = encontra a maioria dos números corretos.
    """
    y_pred_round = K.round(K.clip(y_pred, 0, 1))
    tp = K.sum(y_true * y_pred_round)
    fn = K.sum(y_true * (1 - y_pred_round))
    recall = tp / (tp + fn + K.epsilon())
    return recall


def f1_score_metric(y_true, y_pred):
    """
    Calcula o F1 Score durante o treinamento
    
    F1 = 2 * (Precision * Recall) / (Precision + Recall)
    
    Média harmônica entre Precision e Recall.
    Útil quando precisamos balancear ambas as métricas.
    """
    y_pred_round = K.round(K.clip(y_pred, 0, 1))
    tp = K.sum(y_true * y_pred_round)
    fp = K.sum((1 - y_true) * y_pred_round)
    fn = K.sum(y_true * (1 - y_pred_round))
    
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())
    return f1


# ========================
# Compilar Modelo
# ========================
print("📊 Configurando métricas de monitoramento...")
print("   - Accuracy: Porcentagem de acertos gerais")
print("   - Precision: Quantos positivos preditos são corretos")
print("   - Recall: Quantos positivos reais foram encontrados")
print("   - F1 Score: Média harmônica entre Precision e Recall")
print("   - AUC: Área sob a curva ROC (capacidade de discriminação)")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        precision_metric,
        recall_metric,
        f1_score_metric,
        tf.keras.metrics.AUC(name='auc')
    ]
)

print("\n✅ Modelo compilado com sucesso!")
print("\n📊 Resumo do Modelo:")
model.summary()

---
## 6. 🏋️ Treinamento do Modelo

### Callbacks Utilizados:

1. **ModelCheckpoint** 💾:
   - Salva o modelo **a cada 5 épocas**
   - Salva na pasta `checkpoints/`
   - Formato: `model_epoch_{epoch:03d}_val_loss_{val_loss:.4f}.keras`
   - Permite retomar o treinamento de qualquer ponto

2. **EarlyStopping** 🛑:
   - Monitora `val_loss`
   - Para o treinamento se não houver melhoria por 15 épocas
   - Restaura os melhores pesos

3. **ReduceLROnPlateau** 📉:
   - Reduz o learning rate quando `val_loss` para de melhorar
   - Fator de redução: 0.5
   - Paciência: 10 épocas
   - Learning rate mínimo: 1e-7

### Parâmetros de Treinamento:
- **Épocas**: 200 (máximo)
- **Batch Size**: 32
- **Validation Split**: Dados de validação separados
- **Checkpoints**: Salvos a cada 5 épocas


In [ ]:
# ============================================================================
# CALLBACKS E TREINAMENTO
# ============================================================================

class SaveEveryNEpochs(tf.keras.callbacks.Callback):
    def __init__(self, save_freq=5, filepath='checkpoints/model_epoch_{epoch:03d}.keras'):
        super().__init__()
        self.save_freq = save_freq
        self.filepath = filepath
        
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_freq == 0:
            filepath = self.filepath.format(epoch=epoch+1)
            self.model.save(filepath)
            print(f"\n💾 Modelo salvo: {filepath}")

import os
os.makedirs('checkpoints', exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1),
    SaveEveryNEpochs(save_freq=5)
]

print("✅ Callbacks configurados")

# ============================================================================
# TREINAMENTO COM 5 INPUTS
# ============================================================================
print("\n" + "="*80)
print("🏋️ INICIANDO TREINAMENTO (MODELO ULTRA PROFUNDO)")
print("="*80)
print(f"\n   Épocas: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   5 Inputs:")
print(f"      - Sequência: {X_train_norm.shape}")
print(f"      - Frequência: {X_train_freq.shape}")
print(f"      - Gap: {X_train_gap.shape}")
print(f"      - Top10: {X_train_top10.shape}")
print(f"      - Bottom10: {X_train_bottom10.shape}")
print("\n")

history = model.fit(
    [X_train_norm, X_train_freq, X_train_gap, X_train_top10, X_train_bottom10],
    y_train,
    validation_data=(
        [X_val_norm, X_val_freq, X_val_gap, X_val_top10, X_val_bottom10],
        y_val
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*80)
print("✅ TREINAMENTO CONCLUÍDO!")
print("="*80)


---
## 7. 📊 Visualização dos Resultados do Treinamento

Vamos plotar as métricas de treinamento e validação para entender como o modelo aprendeu:

1. **Loss (Binary Crossentropy)**: Quanto menor, melhor
2. **Accuracy**: Porcentagem de acertos
3. **F1 Score**: Balanço entre precisão e recall


In [ ]:
# Plotar histórico de treinamento
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ==========================================
# Loss
# ==========================================
axes[0].plot(history.history['loss'], label='Treino', linewidth=2, color='#1f77b4')
axes[0].plot(history.history['val_loss'], label='Validação', linewidth=2, color='#ff7f0e')
axes[0].set_title('Loss (Binary Crossentropy)', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ==========================================
# Accuracy
# ==========================================
axes[1].plot(history.history['accuracy'], label='Treino', linewidth=2, color='#2ca02c')
axes[1].plot(history.history['val_accuracy'], label='Validação', linewidth=2, color='#d62728')
axes[1].set_title('Accuracy', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# ==========================================
# F1 Score
# ==========================================
axes[2].plot(history.history['f1_score_metric'], label='Treino', linewidth=2, color='#9467bd')
axes[2].plot(history.history['val_f1_score_metric'], label='Validação', linewidth=2, color='#8c564b')
axes[2].set_title('F1 Score', fontweight='bold', fontsize=14)
axes[2].set_xlabel('Época', fontsize=12)
axes[2].set_ylabel('F1 Score', fontsize=12)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history_resnet.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo como 'training_history_resnet.png'")

---
## 8. 🎯 Avaliação no Conjunto de Teste

Agora vamos avaliar o modelo no conjunto de teste (dados que o modelo nunca viu durante o treinamento).

Isso nos dá uma estimativa realista do desempenho do modelo em dados novos.


In [ ]:
# ============================================================================
# AVALIAÇÃO DO MODELO (5 INPUTS)
# ============================================================================

print("\n" + "="*80)
print("📊 AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*80)

test_results = model.evaluate(
    [X_test_norm, X_test_freq, X_test_gap, X_test_top10, X_test_bottom10],
    y_test,
    verbose=1
)

print("\n📋 Resultados:")
for metric_name, value in zip(model.metrics_names, test_results):
    print(f"   {metric_name}: {value:.4f}")


---
## 9. 🔮 Exemplo de Predição

Vamos fazer uma predição em uma amostra do conjunto de teste para ver como o modelo funciona na prática.

O modelo retorna 60 probabilidades (uma para cada número de 1 a 60). Pegamos os 6 números com maior probabilidade como nossa predição.


In [ ]:
# ============================================================================
# PREDIÇÃO DE EXEMPLO (5 INPUTS)
# ============================================================================

print("\n" + "="*80)
print("🔮 EXEMPLO DE PREDIÇÃO")
print("="*80)

idx = np.random.randint(0, len(X_test))

predictions = model.predict([
    X_test_norm[idx:idx+1],
    X_test_freq[idx:idx+1],
    X_test_gap[idx:idx+1],
    X_test_top10[idx:idx+1],
    X_test_bottom10[idx:idx+1]
], verbose=0)[0]

top_6_indices = np.argsort(predictions)[-6:]
top_6_numbers = top_6_indices + 1
top_6_probs = predictions[top_6_indices]

true_indices = np.where(y_test[idx] == 1)[0]
true_numbers = true_indices + 1

print(f"\n📊 Amostra #{idx}")
print(f"\n🎯 Números Verdadeiros: {sorted(true_numbers)}")
print(f"🔮 Números Previstos:   {sorted(top_6_numbers)}")

matches = len(set(true_numbers) & set(top_6_numbers))
print(f"\n✅ Acertos: {matches}/6")
print("="*80)


---
## 10. 📈 Análise de Probabilidades

Vamos visualizar a distribuição de probabilidades para todos os 60 números.


In [ ]:
# Plotar distribuição de probabilidades
fig, ax = plt.subplots(figsize=(16, 6))

# Criar barras
numbers = np.arange(1, 61)
colors = ['#2ecc71' if i in top_6_numbers else '#3498db' for i in numbers]
bars = ax.bar(numbers, predictions, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)

# Destacar números verdadeiros
for num in true_numbers:
    bars[num-1].set_edgecolor('red')
    bars[num-1].set_linewidth(3)

# Configurações do gráfico
ax.set_xlabel('Número', fontsize=12, fontweight='bold')
ax.set_ylabel('Probabilidade', fontsize=12, fontweight='bold')
ax.set_title('Distribuição de Probabilidades para Todos os Números', fontsize=14, fontweight='bold')
ax.set_xticks(numbers)
ax.set_xticklabels(numbers, fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', alpha=0.7, label='Top 6 Previstos'),
    Patch(facecolor='#3498db', alpha=0.7, label='Outros'),
    Patch(facecolor='white', edgecolor='red', linewidth=3, label='Números Verdadeiros')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig('probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo como 'probability_distribution.png'")

---
## 12. 🎯 Avaliação de Acertos por Quantidade de Números Preditos

Esta seção avalia a performance do modelo considerando diferentes estratégias de predição:

- **Predizendo 6 números**: Quantos dos 6 números sorteados são acertados?
- **Predizendo 7 números**: Se escolhermos os 7 mais prováveis, quantos acertamos?
- **Predizendo 8, 9, 10 números**: Mesma lógica, aumentando a quantidade de números escolhidos

### 💡 Interpretação:
- **Sena**: Acertar todos os 6 números sorteados
- **Quina**: Acertar 5 dos 6 números sorteados
- **Quadra**: Acertar 4 dos 6 números sorteados
- **Terno**: Acertar 3 dos 6 números sorteados

Quanto mais números predizemos, maior a chance de acertar, mas menor a precisão.


In [ ]:
# ============================================================================
# AVALIAÇÃO DE ACERTOS (5 INPUTS)
# ============================================================================

def evaluate_hits_by_prediction_count(model, inputs, y_test, pred_counts=[6, 7, 8, 9, 10]):
    print("="*80)
    print("🎯 AVALIAÇÃO DE ACERTOS")
    print("="*80)
    
    predictions = model.predict(inputs, verbose=0)
    results = {}
    
    for n_pred in pred_counts:
        hits_distribution = {k: 0 for k in range(7)}
        total_hits = 0
        
        for i in range(len(predictions)):
            pred_set = set(np.argsort(predictions[i])[-n_pred:] + 1)
            true_set = set(np.where(y_test[i] == 1)[0] + 1)
            hits = len(pred_set & true_set)
            hits_distribution[hits] += 1
            total_hits += hits
        
        n_samples = len(predictions)
        results[n_pred] = {
            'avg_hits': total_hits / n_samples,
            'distribution': hits_distribution,
            'terno_or_more': sum(hits_distribution[k] for k in [3,4,5,6]) / n_samples * 100
        }
    
    print(f"\n{'Nums':<8} {'Média':<10} {'Terno+':<10}")
    print("-"*30)
    for n_pred in pred_counts:
        r = results[n_pred]
        print(f"{n_pred:<8} {r['avg_hits']:<10.3f} {r['terno_or_more']:<10.2f}%")
    
    return results

# Executar
test_inputs = [X_test_norm, X_test_freq, X_test_gap, X_test_top10, X_test_bottom10]
results = evaluate_hits_by_prediction_count(model, test_inputs, y_test)


In [ ]:
# ============================================================================
# VISUALIZAÇÃO GRÁFICA DOS RESULTADOS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pred_counts = [6, 7, 8, 9, 10]

# Gráfico 1: Média de acertos por quantidade de números preditos
ax1 = axes[0]
avg_hits = [results[n]['avg_hits'] for n in pred_counts]
bars1 = ax1.bar(pred_counts, avg_hits, color='#3498db', edgecolor='black', alpha=0.8)
ax1.set_xlabel('Quantidade de Números Preditos', fontsize=12, fontweight='bold')
ax1.set_ylabel('Média de Acertos (de 6)', fontsize=12, fontweight='bold')
ax1.set_title('📊 Média de Acertos por Quantidade Predita', fontsize=14, fontweight='bold')
ax1.set_xticks(pred_counts)
ax1.set_ylim(0, 6)
ax1.axhline(y=6, color='green', linestyle='--', alpha=0.5, label='Sena (6 acertos)')
ax1.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Quina (5 acertos)')
ax1.axhline(y=4, color='red', linestyle='--', alpha=0.5, label='Quadra (4 acertos)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3, axis='y')

# Adicionar valores nas barras
for bar, val in zip(bars1, avg_hits):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Gráfico 2: Percentual de prêmios
ax2 = axes[1]
x = np.arange(len(pred_counts))
width = 0.2

terno = [results[n]['terno_or_more'] for n in pred_counts]
quadra = [results[n]['quadra_or_more'] for n in pred_counts]
quina = [results[n]['quina_or_more'] for n in pred_counts]
sena = [results[n]['sena_or_more'] for n in pred_counts]

bars_terno = ax2.bar(x - 1.5*width, terno, width, label='Terno+', color='#9b59b6', alpha=0.8)
bars_quadra = ax2.bar(x - 0.5*width, quadra, width, label='Quadra+', color='#e74c3c', alpha=0.8)
bars_quina = ax2.bar(x + 0.5*width, quina, width, label='Quina+', color='#f39c12', alpha=0.8)
bars_sena = ax2.bar(x + 1.5*width, sena, width, label='Sena', color='#27ae60', alpha=0.8)

ax2.set_xlabel('Quantidade de Números Preditos', fontsize=12, fontweight='bold')
ax2.set_ylabel('Percentual de Acertos (%)', fontsize=12, fontweight='bold')
ax2.set_title('🏆 Percentual de Prêmios por Quantidade Predita', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(pred_counts)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('evaluation_hits.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráfico salvo como 'evaluation_hits.png'")

# Resumo final
print("\n" + "="*80)
print("📋 RESUMO FINAL")
print("="*80)
print(f"\n• Melhor estratégia para TERNO ou mais: Predizer {pred_counts[np.argmax(terno)]} números ({max(terno):.2f}%)")
print(f"• Melhor estratégia para QUADRA ou mais: Predizer {pred_counts[np.argmax(quadra)]} números ({max(quadra):.2f}%)")
print(f"• Melhor estratégia para QUINA ou mais: Predizer {pred_counts[np.argmax(quina)]} números ({max(quina):.2f}%)")
print(f"• Melhor estratégia para SENA: Predizer {pred_counts[np.argmax(sena)]} números ({max(sena):.2f}%)")
print("="*80)


---
## 13. 🎲 Comparação com Aleatoriedade (Baseline Aleatório)

### Por que esta análise é importante?

Para validar se o modelo está realmente aprendendo padrões, precisamos comparar seu desempenho com o que seria esperado por **puro acaso**.

### 📊 Distribuição Hipergeométrica

A probabilidade de acertar exatamente `k` números ao escolher `n` números de um universo de 60, sendo 6 os números sorteados, segue a **distribuição hipergeométrica**:

$$P(X = k) = \frac{\binom{6}{k} \cdot \binom{54}{n-k}}{\binom{60}{n}}$$

Onde:
- `k` = quantidade de acertos
- `n` = quantidade de números escolhidos
- 6 = números sorteados
- 54 = números não sorteados
- 60 = total de números

### 🎯 O que queremos ver?
- **Modelo > Aleatoriedade**: O modelo está aprendendo padrões!
- **Modelo = Aleatoriedade**: O modelo não aprendeu nada útil
- **Modelo < Aleatoriedade**: Algo está errado com o modelo


In [ ]:
# ============================================================================
# COMPARAÇÃO COM ALEATORIEDADE
# ============================================================================
from scipy.stats import hypergeom

def calculate_random_baseline(n_chosen, total_numbers=60, drawn_numbers=6):
    rv = hypergeom(total_numbers, drawn_numbers, n_chosen)
    probs = {}
    for k in range(min(n_chosen, drawn_numbers) + 1):
        probs[k] = rv.pmf(k) * 100
    return probs, rv.mean()


def compare_model_vs_random(model, X_seq, X_freq, X_gap, y_test, pred_counts=[6, 7, 8, 9, 10]):
    print("="*80)
    print("🎲 COMPARAÇÃO: MODELO vs ALEATORIEDADE")
    print("="*80)
    
    predictions = model.predict([X_seq, X_freq, X_gap], verbose=0)
    n_samples = len(predictions)
    
    comparison_results = {}
    
    for n_pred in pred_counts:
        random_probs, random_expected = calculate_random_baseline(n_pred)
        
        model_hits = {k: 0 for k in range(7)}
        total_hits = 0
        
        for i in range(len(predictions)):
            top_n_indices = np.argsort(predictions[i])[-n_pred:]
            predicted_numbers = set(top_n_indices + 1)
            true_indices = np.where(y_test[i] == 1)[0]
            true_numbers = set(true_indices + 1)
            
            hits = len(predicted_numbers & true_numbers)
            model_hits[hits] += 1
            total_hits += hits
        
        model_expected = total_hits / n_samples
        improvement = ((model_expected - random_expected) / random_expected) * 100
        
        comparison_results[n_pred] = {
            'random_expected': random_expected,
            'model_expected': model_expected,
            'improvement': improvement,
            'random_probs': random_probs,
            'model_probs': {k: v/n_samples*100 for k, v in model_hits.items()}
        }
    
    print("\n" + "-"*80)
    print(f"{'Números':<12} {'Aleatorio':<15} {'Modelo':<15} {'Melhoria':<15} {'Status'}")
    print("-"*80)
    
    for n_pred in pred_counts:
        r = comparison_results[n_pred]
        status = "EXCELENTE!" if r['improvement'] > 10 else "BOM!" if r['improvement'] > 5 else "OK" if r['improvement'] > 0 else "NEUTRO"
        print(f"{n_pred:<12} {r['random_expected']:<15.4f} {r['model_expected']:<15.4f} {r['improvement']:+14.2f}%  {status}")
    
    print("-"*80)
    return comparison_results


print("\n")
comparison_results = compare_model_vs_random(
    model,
    X_test_norm,
    X_test_freq,
    X_test_gap,
    y_test
)


In [ ]:
# ============================================================================
# VISUALIZAÇÃO: MODELO vs ALEATORIEDADE
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

pred_counts = [6, 7, 8, 9, 10]

# Gráfico 1: Média de acertos - Modelo vs Aleatório
ax1 = axes[0, 0]
x = np.arange(len(pred_counts))
width = 0.35

random_means = [comparison_results[n]['random_expected'] for n in pred_counts]
model_means = [comparison_results[n]['model_expected'] for n in pred_counts]

bars1 = ax1.bar(x - width/2, random_means, width, label='Aleatorio', color='#e74c3c', alpha=0.8)
bars2 = ax1.bar(x + width/2, model_means, width, label='Modelo', color='#27ae60', alpha=0.8)

ax1.set_xlabel('Quantidade de Numeros Preditos', fontsize=12, fontweight='bold')
ax1.set_ylabel('Media de Acertos', fontsize=12, fontweight='bold')
ax1.set_title('Media de Acertos: Modelo vs Aleatoriedade', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(pred_counts)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Adicionar valores
for bar, val in zip(bars1, random_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, model_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Gráfico 2: Melhoria percentual
ax2 = axes[0, 1]
improvements = [comparison_results[n]['improvement'] for n in pred_counts]
colors = ['#27ae60' if imp > 0 else '#e74c3c' for imp in improvements]

bars3 = ax2.bar(pred_counts, improvements, color=colors, alpha=0.8, edgecolor='black')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=2)
ax2.axhline(y=5, color='green', linestyle='--', alpha=0.5, label='Bom (+5%)')
ax2.axhline(y=10, color='darkgreen', linestyle='--', alpha=0.5, label='Excelente (+10%)')
ax2.axhline(y=-5, color='red', linestyle='--', alpha=0.5, label='Problema (-5%)')

ax2.set_xlabel('Quantidade de Numeros Preditos', fontsize=12, fontweight='bold')
ax2.set_ylabel('Melhoria sobre Aleatoriedade (%)', fontsize=12, fontweight='bold')
ax2.set_title('Melhoria do Modelo sobre o Acaso', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars3, improvements):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5 if val >= 0 else bar.get_height() - 1, 
             f'{val:+.2f}%', ha='center', va='bottom' if val >= 0 else 'top', fontsize=11, fontweight='bold')

# Gráfico 3: Comparação detalhada para 6 números
ax3 = axes[1, 0]
n_pred = 6
r = comparison_results[n_pred]

hits_range = list(range(7))
random_probs = [r['random_probs'].get(k, 0) for k in hits_range]
model_probs = [r['model_probs'].get(k, 0) for k in hits_range]

x3 = np.arange(len(hits_range))
bars4 = ax3.bar(x3 - width/2, random_probs, width, label='Aleatorio', color='#e74c3c', alpha=0.8)
bars5 = ax3.bar(x3 + width/2, model_probs, width, label='Modelo', color='#27ae60', alpha=0.8)

ax3.set_xlabel('Quantidade de Acertos', fontsize=12, fontweight='bold')
ax3.set_ylabel('Probabilidade (%)', fontsize=12, fontweight='bold')
ax3.set_title(f'Distribuicao de Acertos (Predizendo {n_pred} numeros)', fontsize=14, fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(hits_range)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Gráfico 4: Comparação detalhada para 10 números
ax4 = axes[1, 1]
n_pred = 10
r = comparison_results[n_pred]

hits_range = list(range(7))
random_probs = [r['random_probs'].get(k, 0) for k in hits_range]
model_probs = [r['model_probs'].get(k, 0) for k in hits_range]

bars6 = ax4.bar(x3 - width/2, random_probs, width, label='Aleatorio', color='#e74c3c', alpha=0.8)
bars7 = ax4.bar(x3 + width/2, model_probs, width, label='Modelo', color='#27ae60', alpha=0.8)

ax4.set_xlabel('Quantidade de Acertos', fontsize=12, fontweight='bold')
ax4.set_ylabel('Probabilidade (%)', fontsize=12, fontweight='bold')
ax4.set_title(f'Distribuicao de Acertos (Predizendo {n_pred} numeros)', fontsize=14, fontweight='bold')
ax4.set_xticks(x3)
ax4.set_xticklabels(hits_range)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_vs_random.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGrafico salvo como 'model_vs_random.png'")

# Conclusão
print("\n" + "="*80)
print("CONCLUSAO: O MODELO ESTA SE DESCOLANDO DA ALEATORIEDADE?")
print("="*80)

avg_improvement = np.mean([comparison_results[n]['improvement'] for n in pred_counts])

if avg_improvement > 10:
    conclusion = "SIM! O modelo esta SIGNIFICATIVAMENTE acima do acaso."
    emoji = "EXCELENTE"
elif avg_improvement > 5:
    conclusion = "SIM! O modelo esta acima do acaso, indicando aprendizado."
    emoji = "BOM"
elif avg_improvement > 0:
    conclusion = "LEVEMENTE. O modelo mostra sinais de aprendizado."
    emoji = "OK"
elif avg_improvement > -5:
    conclusion = "NAO. O modelo esta similar ao acaso."
    emoji = "NEUTRO"
else:
    conclusion = "NAO. O modelo esta ABAIXO do acaso (possivel problema)."
    emoji = "PROBLEMA"

print(f"\nMelhoria media sobre aleatoriedade: {avg_improvement:+.2f}%")
print(f"\nStatus: [{emoji}]")
print(f"\n{conclusion}")
print("="*80)


---
## 14. 📊 Comparação de Estratégias: Modelo vs Frequência vs Aleatório

### Estratégias Comparadas:

| Estratégia | Descrição |
|------------|----------|
| **Aleatório** | Baseline teórico usando distribuição hipergeométrica |
| **Rede Neural** | Predições do modelo treinado |
| **Mais Frequentes** | 6 números que mais apareceram na janela de input |
| **Menos Frequentes** | 6 números que menos apareceram na janela de input |
| **Mix (3+3)** | 3 mais frequentes + 3 menos frequentes |

### 💡 Por que essa comparação é importante?
- Se o modelo não supera heurísticas simples de frequência, ele não aprendeu nada útil
- Permite identificar qual estratégia funciona melhor para a Mega-Sena
- Testa a hipótese de "números quentes" vs "números frios"


In [ ]:
# ============================================================================
# COMPARAÇÃO DE ESTRATÉGIAS: MODELO vs FREQUÊNCIA vs ALEATÓRIO
# ============================================================================

def get_frequency_based_predictions(X_data, n_most=6, n_least=0):
    predictions = []
    for i in range(len(X_data)):
        freq = np.zeros(60)
        all_numbers = X_data[i].flatten().astype(int)
        for num in all_numbers:
            if 1 <= num <= 60:
                freq[num - 1] += 1
        
        sorted_indices = np.argsort(freq)
        selected = set()
        
        if n_most > 0:
            most_freq_indices = sorted_indices[-n_most:]
            selected.update(most_freq_indices + 1)
        
        if n_least > 0:
            least_freq_indices = sorted_indices[:n_least]
            selected.update(least_freq_indices + 1)
        
        predictions.append(selected)
    return predictions


def evaluate_strategy(predictions_list, y_test, strategy_name):
    hits_distribution = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}
    total_hits = 0
    
    for i in range(len(predictions_list)):
        predicted_numbers = predictions_list[i]
        true_indices = np.where(y_test[i] == 1)[0]
        true_numbers = set(true_indices + 1)
        
        hits = len(predicted_numbers & true_numbers)
        hits_distribution[hits] += 1
        total_hits += hits
    
    n_samples = len(predictions_list)
    avg_hits = total_hits / n_samples
    
    return {
        'name': strategy_name,
        'avg_hits': avg_hits,
        'distribution': hits_distribution,
        'sena_pct': hits_distribution[6] / n_samples * 100,
        'quina_pct': hits_distribution[5] / n_samples * 100,
        'quadra_pct': hits_distribution[4] / n_samples * 100,
        'terno_pct': hits_distribution[3] / n_samples * 100,
        'terno_or_more': (hits_distribution[3] + hits_distribution[4] + 
                         hits_distribution[5] + hits_distribution[6]) / n_samples * 100
    }


def compare_all_strategies(model, X_seq, X_freq, X_gap, X_raw, y_test):
    print("="*90)
    print("📊 COMPARAÇÃO DE ESTRATÉGIAS")
    print("="*90)
    
    # Rede Neural
    print("\n⏳ Rede Neural...")
    nn_predictions_raw = model.predict([X_seq, X_freq, X_gap], verbose=0)
    nn_predictions = [set(np.argsort(p)[-6:] + 1) for p in nn_predictions_raw]
    nn_results = evaluate_strategy(nn_predictions, y_test, "Rede Neural")
    
    # Frequência
    print("⏳ 6 Mais Frequentes...")
    most_freq = get_frequency_based_predictions(X_raw, n_most=6, n_least=0)
    most_freq_results = evaluate_strategy(most_freq, y_test, "6 Mais Frequentes")
    
    print("⏳ 6 Menos Frequentes...")
    least_freq = get_frequency_based_predictions(X_raw, n_most=0, n_least=6)
    least_freq_results = evaluate_strategy(least_freq, y_test, "6 Menos Frequentes")
    
    print("⏳ Mix (3+3)...")
    mix = get_frequency_based_predictions(X_raw, n_most=3, n_least=3)
    mix_results = evaluate_strategy(mix, y_test, "Mix (3+3)")
    
    # Aleatório
    random_probs, random_expected = calculate_random_baseline(6)
    random_results = {
        'name': 'Aleatório',
        'avg_hits': random_expected,
        'sena_pct': random_probs.get(6, 0),
        'quina_pct': random_probs.get(5, 0),
        'quadra_pct': random_probs.get(4, 0),
        'terno_pct': random_probs.get(3, 0),
        'terno_or_more': sum(random_probs.get(k, 0) for k in [3, 4, 5, 6])
    }
    
    all_results = [random_results, nn_results, most_freq_results, least_freq_results, mix_results]
    
    # Tabela
    print("\n" + "="*90)
    print(f"{'Estratégia':<22} {'Média':<10} {'Sena':<10} {'Quina':<10} {'Quadra':<10} {'Terno+':<10}")
    print("-"*90)
    
    for r in all_results:
        print(f"{r['name']:<22} {r['avg_hits']:<10.4f} {r['sena_pct']:<10.4f}% {r['quina_pct']:<10.4f}% {r['quadra_pct']:<10.4f}% {r['terno_or_more']:<10.2f}%")
    
    print("-"*90)
    
    # Ranking
    print("\n🏆 RANKING:")
    sorted_results = sorted(all_results, key=lambda x: x['avg_hits'], reverse=True)
    for i, r in enumerate(sorted_results):
        medal = "🥇" if i == 0 else "🥈" if i == 1 else "🥉" if i == 2 else "  "
        improvement = ((r['avg_hits'] - random_expected) / random_expected) * 100
        print(f"{medal} {i+1}º {r['name']:<20} - Média: {r['avg_hits']:.4f} ({improvement:+.2f}% vs aleatório)")
    
    return all_results


print("\n")
strategy_results = compare_all_strategies(
    model,
    X_test_norm,
    X_test_freq,
    X_test_gap,
    X_test,
    y_test
)


In [ ]:
# ============================================================================
# VISUALIZAÇÃO: COMPARAÇÃO DE ESTRATÉGIAS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dados para os gráficos
strategies = [r['name'] for r in strategy_results]
avg_hits = [r['avg_hits'] for r in strategy_results]
terno_or_more = [r['terno_or_more'] for r in strategy_results]

# Cores diferentes para cada estratégia
colors = ['#95a5a6', '#27ae60', '#3498db', '#e74c3c', '#9b59b6']

# Gráfico 1: Média de Acertos
ax1 = axes[0]
bars1 = ax1.bar(strategies, avg_hits, color=colors, edgecolor='black', alpha=0.8)

# Linha de referência para aleatório
random_baseline = strategy_results[0]['avg_hits']
ax1.axhline(y=random_baseline, color='red', linestyle='--', linewidth=2, label='Baseline Aleatorio')

ax1.set_xlabel('Estrategia', fontsize=12, fontweight='bold')
ax1.set_ylabel('Media de Acertos', fontsize=12, fontweight='bold')
ax1.set_title('Media de Acertos por Estrategia', fontsize=14, fontweight='bold')
ax1.set_xticklabels(strategies, rotation=15, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Adicionar valores nas barras
for bar, val in zip(bars1, avg_hits):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Gráfico 2: Probabilidade de Terno ou mais
ax2 = axes[1]
bars2 = ax2.bar(strategies, terno_or_more, color=colors, edgecolor='black', alpha=0.8)

# Linha de referência para aleatório
random_terno = strategy_results[0]['terno_or_more']
ax2.axhline(y=random_terno, color='red', linestyle='--', linewidth=2, label='Baseline Aleatorio')

ax2.set_xlabel('Estrategia', fontsize=12, fontweight='bold')
ax2.set_ylabel('Probabilidade (%)', fontsize=12, fontweight='bold')
ax2.set_title('Probabilidade de Terno ou Mais (3+ acertos)', fontsize=14, fontweight='bold')
ax2.set_xticklabels(strategies, rotation=15, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Adicionar valores nas barras
for bar, val in zip(bars2, terno_or_more):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
             f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('strategies_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGrafico salvo como 'strategies_comparison.png'")

# ========================================
# Conclusão Final
# ========================================
print("\n" + "="*90)
print("📋 CONCLUSÃO FINAL")
print("="*90)

# Encontrar a melhor estratégia
best_strategy = max(strategy_results, key=lambda x: x['avg_hits'])
worst_strategy = min(strategy_results, key=lambda x: x['avg_hits'])

print(f"\n🏆 Melhor estratégia: {best_strategy['name']}")
print(f"   Média de acertos: {best_strategy['avg_hits']:.4f}")
print(f"   Terno ou mais: {best_strategy['terno_or_more']:.2f}%")

print(f"\n❌ Pior estratégia: {worst_strategy['name']}")
print(f"   Média de acertos: {worst_strategy['avg_hits']:.4f}")

# Verificar se NN é a melhor
nn_result = next(r for r in strategy_results if r['name'] == 'Rede Neural')
if nn_result == best_strategy:
    print("\n✅ A REDE NEURAL É A MELHOR ESTRATÉGIA!")
    print("   O modelo aprendeu padrões úteis que superam heurísticas simples.")
else:
    nn_rank = sorted(strategy_results, key=lambda x: x['avg_hits'], reverse=True).index(nn_result) + 1
    print(f"\n⚠️ A Rede Neural ficou em {nn_rank}º lugar.")
    print(f"   Considere melhorar a arquitetura ou hiperparâmetros.")

print("\n" + "="*90)


---
## 11. 🎓 Conclusão e Próximos Passos

### 📝 Resumo do Modelo:
- ✅ Implementamos blocos **ResNet** para melhor extração de features
- ✅ Utilizamos **LSTM** para capturar dependências temporais
- ✅ Criamos uma **torre densa profunda** para classificação
- ✅ Aplicamos **regularização L2** e **Dropout** para evitar overfitting
- ✅ Usamos **callbacks** para otimizar o treinamento

### 🚀 Possíveis Melhorias:
1. **Attention Mechanism**: Adicionar camadas de atenção após LSTM
2. **Ensemble**: Combinar múltiplos modelos
3. **Feature Engineering**: Adicionar features estatísticas
4. **Hyperparameter Tuning**: Otimizar hiperparâmetros com Optuna
5. **Data Augmentation**: Aumentar dados de treinamento

### ⚠️ Importante:
Este modelo é para fins **educacionais** e de **pesquisa**. A Mega-Sena é um jogo de azar e os resultados são aleatórios. Nenhum modelo pode prever com certeza os números sorteados.

### 📚 Referências:
- [ResNet Paper](https://arxiv.org/abs/1512.03385)
- [LSTM Paper](https://www.bioinf.jku.at/publications/older/2604.pdf)
- [Keras Documentation](https://keras.io/)


---
## 15. 🚀 Backtest Estendido e Predição do Próximo Jogo

Atendendo à solicitação:
1. **Backtest na Janela de 2000 Jogos**: Vamos avaliar a estratégia de jogar os **8 melhores números** nos últimos 2000 concursos.
2. **Predição do Futuro**: Vamos prever o **próximo concurso** (inédito) e exibir os 8 números mais prováveis.


In [ ]:
# ============================================================================
# 1. CARREGAR DADOS BRUTOS (MEGA-SENA.XLSX)
# ============================================================================
print("="*80)
print("📥 CARREGANDO DADOS COMPLETOS PARA BACKTEST (2000 JOGOS)")
print("="*80)

try:
    df = pd.read_excel('Mega-Sena.xlsx')
    
    # Identificar colunas de bolas
    ball_columns = []
    for col in df.columns:
        if 'bola' in col.lower() or 'dezena' in col.lower():
            ball_columns.append(col)
            
    if len(ball_columns) != 6:
         # Tentar identificar por valor
        ball_columns = []
        for col in df.columns:
            numeric_col = pd.to_numeric(df[col], errors='coerce')
            if numeric_col.notna().sum() > 0:
                if numeric_col.min() >= 1 and numeric_col.max() <= 60:
                    ball_columns.append(col)
        ball_columns = ball_columns[:6]

    # Preparar dados ordenados
    data_balls = df[ball_columns].dropna().astype(int)
    data_full = np.sort(data_balls.values, axis=1)
    
    print(f"✅ Total de jogos carregados: {len(data_full)}")
    print(f"   Colunas identificadas: {ball_columns}")
    
except Exception as e:
    print(f"❌ Erro ao ler Excel: {e}")
    # Fallback para data_raw se disponível do prepare_data (não garantido aqui)
    raise

# ============================================================================
# 2. DEFINIR FUNÇÃO PARA GERAR INPUTS ON-THE-FLY
# ============================================================================
# Precisamos funções auxiliares pois vamos processar uma janela deslizante personalizada

def prepare_single_input(sequence_window):
    # sequence_window: shape (window_size, 6)
    # Retorna os 5 inputs normalizados para o modelo
    
    # 1. Norm
    seq_norm = (sequence_window / 60.0).astype(np.float32)
    
    # 2. Freq
    freq = np.zeros(60, dtype=np.float32)
    all_nums = sequence_window.flatten().astype(int)
    for num in all_nums:
        if 1 <= num <= 60:
            freq[num-1] += 1
    freq_norm = freq / 60.0
    
    # 3. Gap
    window_sz = len(sequence_window)
    total_nums = window_sz * 6
    expected = total_nums / 60.0
    gap = expected - freq
    gap_norm = gap / expected
    
    # 4. Top/Bottom 10
    sorted_idx = np.argsort(freq)
    top10 = np.zeros(60, dtype=np.float32)
    bottom10 = np.zeros(60, dtype=np.float32)
    top10[sorted_idx[-10:]] = 1.0
    bottom10[sorted_idx[:10]] = 1.0
    
    # Expandir dims para (1, ...)
    return (
        np.expand_dims(seq_norm, 0),
        np.expand_dims(freq_norm, 0),
        np.expand_dims(gap_norm, 0),
        np.expand_dims(top10, 0),
        np.expand_dims(bottom10, 0)
    )

# ============================================================================
# 3. BACKTEST: JANELA DE 2000 JOGOS
# ============================================================================
# Avaliar os últimos 2000 jogos (ou o máximo possível)
test_window = 2000
total_games = len(data_full)
# Usar o mesmo WINDOW_SIZE usado no treinamento
# Se não estiver definido, tentamos inferir
try:
    win_size = WINDOW_SIZE
except NameError:
    win_size = 100 # Valor padrao se nao definido
    print(f"⚠️ WINDOW_SIZE não encontrado, usando {win_size}")

start_idx = max(win_size, total_games - test_window)

print(f"\n⏳ Iniciando Backtest nos últimos {total_games - start_idx} jogos...")
print(f"   Intervalo: Jogo {start_idx} até {total_games-1}")
print(f"   Window Size: {win_size}")

hits_8_numbers = 0
total_eval = 0
distribution_8 = {0:0, 1:0, 2:0, 3:0, 4:0, 5:0, 6:0}

X_seq_list = []
X_freq_list = []
X_gap_list = []
X_top10_list = []
X_bot10_list = []
y_true_list = []

indices_to_test = range(start_idx, total_games)

for i in indices_to_test:
    # Janela anterior
    window_data = data_full[i-win_size : i]
    target = data_full[i]
    
    # Inputs
    inputs = prepare_single_input(window_data)
    
    X_seq_list.append(inputs[0][0])
    X_freq_list.append(inputs[1][0])
    X_gap_list.append(inputs[2][0])
    X_top10_list.append(inputs[3][0])
    X_bot10_list.append(inputs[4][0])
    y_true_list.append(target)

# Converter para numpy
X_seq_arr = np.array(X_seq_list)
X_freq_arr = np.array(X_freq_list)
X_gap_arr = np.array(X_gap_list)
X_top10_arr = np.array(X_top10_list)
X_bot10_arr = np.array(X_bot10_list)

print("   ✅ Inputs preparados. Executando predição em batch...")

predictions = model.predict([X_seq_arr, X_freq_arr, X_gap_arr, X_top10_arr, X_bot10_arr], verbose=1)

# Analisar Top 8
for i, pred_probs in enumerate(predictions):
    # 8 números mais prováveis
    top_8_idx = np.argsort(pred_probs)[-8:]
    top_8_nums = set(top_8_idx + 1)
    
    true_nums = set(y_true_list[i])
    
    hits = len(top_8_nums & true_nums)
    if hits > 6: hits = 6 # Cap at 6 for display issues if any (but hits max is 6)
    distribution_8[hits] += 1
    total_eval += 1

# Relatório
print("\n" + "="*80)
print(f"📊 RESULTADO DO BACKTEST (JOGANDO 8 NÚMEROS) - {total_eval} JOGOS")
print("="*80)

avg_hits = sum(k*v for k,v in distribution_8.items()) / total_eval
sena = distribution_8[6]
quina = distribution_8[5]
quadra = distribution_8[4]
terno = distribution_8[3]

print(f"\n🔢 Média de Acertos: {avg_hits:.4f}")
print(f"\n🏆 Prêmios (em {total_eval} jogos):")
print(f"   - SENA (6 acertos): {sena:<5} ({sena/total_eval*100:.2f}%)")
print(f"   - QUINA (5 acertos): {quina:<5} ({quina/total_eval*100:.2f}%)")
print(f"   - QUADRA (4 acertos): {quadra:<5} ({quadra/total_eval*100:.2f}%)")
print(f"   - TERNO (3 acertos): {terno:<5} ({terno/total_eval*100:.2f}%)")

print("\n📉 Distribuição Detalhada:")
for k in range(6, -1, -1):
    print(f"   {k} acertos: {distribution_8[k]}")

# ============================================================================
# 4. PREDIÇÃO DO PRÓXIMO JOGO (FUTURO)
# ============================================================================

print("\n" + "="*80)
print(f"🔮 PREDIÇÃO PARA O PRÓXIMO CONCURSO ({len(data_full) + 1})")
print("="*80)

# Pegar a última janela disponível
last_window = data_full[-win_size:]
next_inputs = prepare_single_input(last_window)

next_pred = model.predict(list(next_inputs), verbose=0)[0]

# Top 8 para o próximo jogo
top_8_idx = np.argsort(next_pred)[-8:]
top_8_nums = sorted(top_8_idx + 1)
top_8_probs = next_pred[top_8_idx]

# Top 6 para comparação
top_6_nums = sorted(np.argsort(next_pred)[-6:] + 1)

print(f"\n🎲 Último Jogo ({len(data_full)}): {list(data_full[-1])}")
print(f"\n🔥 OS 8 NÚMEROS MAIS PROVÁVEIS PARA O PRÓXIMO JOGO:")
print(f"   👉 {top_8_nums}")

print(f"\n💎 Aposta Sugerida (6 Números):")
print(f"   👉 {top_6_nums}")

print("="*80)
